# Effect Size — Cohen's d

**DS4DH · Module 04 — Statistical Inference**

*Technique:* Standardised effect size, and why significance and magnitude are different questions

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Sagaustus/ds4dh-colab-pack/blob/main/notebooks/04c_cohens_d.ipynb)

Data: `merged_dataset.csv` — from the `data/` folder of this pack.

---

In [ ]:
# Setup — run this first.
import os, warnings
warnings.filterwarnings('ignore')
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy import stats

# This notebook reads the CSVs sitting next to it. In Colab, upload them from
# the pack's data/ folder when prompted. The exists() guard means a re-run
# part-way through a session will not ask you to upload all over again.
NEEDED = ['merged_dataset.csv']
missing = [f for f in NEEDED if not os.path.exists(f)]
if missing:
    try:
        from google.colab import files
        print('Upload from the data/ folder of the pack: ' + ', '.join(missing))
        files.upload()
    except ImportError:
        raise SystemExit('Place these next to the notebook: ' + ', '.join(missing))

df       = pd.read_csv('merged_dataset.csv')
CITIES = ['Montréal', 'Toronto', 'Edmonton', 'Vancouver']

plt.rcParams['figure.figsize'] = (10, 5.5)
plt.rcParams['axes.grid'] = True
plt.rcParams['grid.alpha'] = 0.25

print(f'Loaded. df has {len(df):,} rows and {df.shape[1]} columns.')

## What this notebook does

A p-value answers "could this be noise?". It does not answer "is this big?".

Those come apart in both directions. A trivial difference measured on a huge
sample gets a tiny p-value. A large difference measured on a small sample gets a
large one. Reporting only p conflates the two.

**Cohen's d** is the difference expressed in standard deviations, which makes it
comparable across variables and sample sizes.

In [ ]:
csd = df.dropna(subset=['csd_code'])

imm = csd[csd['immigrant_status'] == 'Immigrant'][
    ['csd_code', 'geography_name', 'cma', 'Renter']].copy()
imm.columns = ['csd_code', 'geography_name', 'cma', 'renter_stir_imm']

nim = csd[csd['immigrant_status'] == 'Non-immigrants'][['csd_code', 'Renter']].copy()
nim.columns = ['csd_code', 'renter_stir_nim']

penalty_df = imm.merge(nim, on='csd_code', how='inner')
penalty_df['penalty'] = penalty_df['renter_stir_imm'] - penalty_df['renter_stir_nim']
penalty_df = penalty_df.dropna(subset=['penalty'])
penalty_df = penalty_df[penalty_df['cma'].isin(CITIES)]

print(f'{len(penalty_df)} CSDs where BOTH groups have a reported renter STIR')
print()
print(penalty_df[['geography_name', 'cma', 'renter_stir_imm',
                  'renter_stir_nim', 'penalty']].head(8).to_string(index=False))

In [ ]:
def cohens_d_onesample(x):
    """Mean difference in standard-deviation units, for paired differences."""
    return x.mean() / x.std(ddof=1)

print(f'{"City":<12}{"n":>5}{"mean gap":>11}{"sd":>8}{"p":>10}{"d":>9}{"size":>12}')
print('-' * 67)
for city in CITIES:
    s = penalty_df[penalty_df['cma'] == city]['penalty']
    _, p = stats.ttest_1samp(s, 0)
    d = cohens_d_onesample(s)
    a = abs(d)
    label = 'negligible' if a < 0.2 else 'small' if a < 0.5 else 'medium' if a < 0.8 else 'large'
    print(f'{city:<12}{len(s):>5}{s.mean():>+11.2f}{s.std():>8.2f}{p:>10.4f}'
          f'{d:>+9.3f}{label:>12}')

## Reading d

Cohen's conventional bands — 0.2 small, 0.5 medium, 0.8 large — are rules of
thumb from psychology, not laws. They are useful for orientation and should not
be quoted as if they settled anything.

Edmonton's d is around −0.93: large, and negative. Its p-value and its effect
size agree, which is the comfortable case. The other three cities have negligible
effects *and* large p-values — also comfortable, in the opposite direction.

In [ ]:
# The uncomfortable cases: when p and d disagree. Simulate both.
rng = np.random.default_rng(7)

tiny_effect_huge_n = rng.normal(0.05, 1.0, 20000)
big_effect_tiny_n = rng.normal(0.90, 1.0, 8)

for name, x in [('tiny effect, n=20000', tiny_effect_huge_n),
                ('big effect,  n=8', big_effect_tiny_n)]:
    _, p = stats.ttest_1samp(x, 0)
    d = x.mean() / x.std(ddof=1)
    print(f'{name:<24} p={p:<10.5f} d={d:+.3f}')
print()
print('The first is significant and trivial. The second is substantial and')
print('unproven. Neither is reportable on p alone.')

### 🔧 Your turn 1

Change `20000` to `200` in the first simulation and re-run.

The effect size barely moves; the p-value moves a great deal. Which of the two is
a property of the world, and which is a property of your sample size?

## Effect size in the units people think in

d is dimensionless, which makes it comparable but not intuitive. For a policy
audience, convert back to percentage points and then to money.

In [ ]:
ed = penalty_df[penalty_df['cma'] == 'Edmonton']['penalty']
d = cohens_d_onesample(ed)

# Confidence interval for the mean difference.
ci = stats.t.interval(0.95, len(ed) - 1, loc=ed.mean(),
                      scale=stats.sem(ed))

print(f'Edmonton, n = {len(ed)}')
print(f'  mean gap        {ed.mean():+.2f} pp')
print(f'  95% CI          [{ci[0]:+.2f}, {ci[1]:+.2f}] pp')
print(f'  Cohen\'s d       {d:+.3f}')
print()
inc = penalty_df['rent_income'].median() if 'rent_income' in penalty_df else np.nan
print('The CI is what a policy reader needs: it does not include zero, and it')
print('bounds how large the effect could plausibly be in either direction.')

In [ ]:
fig, ax = plt.subplots(figsize=(9, 5))
data = [penalty_df[penalty_df['cma'] == c]['penalty'] for c in CITIES]
ax.boxplot(data, showmeans=True)
ax.set_xticklabels(CITIES)
ax.axhline(0, color='#E8663D', lw=1.5, ls='--', label='no difference')
ax.set_ylabel('Within-CSD gap, immigrant − non-immigrant (pp)')
ax.set_title("Where the effect sizes come from")
ax.legend()
plt.tight_layout()
plt.show()

### 🔧 Your turn 2

Look at the boxplot. Edmonton's box sits below the zero line; the other three
straddle it.

Write the one sentence you would put under this chart in a policy brief. It must
mention the direction, the effect size, and the fact that it applies to one city.

<details markdown="1">
<summary><b>What you should have seen</b> — click to expand</summary>

**Your turn 1.** The effect size is a property of the world (as estimated); the
p-value is a property of the world *and* your sample size. Increasing n shrinks
the p-value without changing d at all. This is why "statistically significant" is
not a synonym for "large", and why a study with enough data can make almost
anything significant.

**Your turn 2.** Something like:

> In Edmonton's 13 census subdivisions with comparable data, immigrant renter
> households spend on average 3.2 percentage points *less* of their income on
> shelter than non-immigrant renters in the same municipalities (95% CI −5.3 to
> −1.1; Cohen's d = −0.93). No comparable difference is detectable in Montréal,
> Toronto or Vancouver.

Note what it does not say: nothing about causes, nothing about other cities,
nothing about change over time. Notebook 11a is about holding that line.

</details>

## Where this stops

You can now say whether a difference is real and whether it is large. You cannot
say it is not simply the effect of *which city* people live in — which is what
regression, in Module 05, is for.